# TP – Séance 3 

**Cours : Real-Time Data Engineering**  
**Enseignant : Dr. Khalil Haddaoui**



# **Moteur de streaming : Temporalité • Watermarks • Fenêtres • State • Backpressure**
#
#
# 🎯 **Objectif du TP**
#
# À la fin de ce TP, vous ne devez plus voir un flux comme une suite de lignes,
# mais comme un **système vivant** qui maintient en continu une cohérence fragile entre :
#
# - **temps métier** (Event Time),
# - **temps machine** (Processing Time),
# - **état** (state) par clé,
# - **pression** (backpressure),
# - **compromis** précision / latence / coût.
#
# 🧠 **Déclic d’entrée**
#
# > Kafka mémorise le passé.  
# > Le moteur de streaming décide du présent.
#
# ⚠️ **Posture attendue**
#
# Ce TP n’évalue pas votre capacité à “utiliser Spark”.
# Il évalue votre capacité à **observer, douter, diagnostiquer et décider**.
#
# Un pipeline peut :
# - mentir sans bug,
# - ralentir sans “erreur”,
# - devenir instable sans être “cassé”.
#
#
# ✅ **Livrables (à rendre)**
#
# 1) Notebook complété (avec vos réponses)  
# 2) 3 captures d’écran : (i) divergence PT vs ET, (ii) corrections late events, (iii) symptôme de backpressure/retard  
# 3) Une page max : **conception finale** (challenge) avec justification des choix


## Charte de lecture (toujours la même) :
#
# - 🎯 Objectif : ce que vous devez comprendre
# - 🧠 Intuition : l’idée simple qui guide
# - 🔁 Rappel : connaissance à réactiver
# - ✅ À faire : action concrète
# - 🔬 À observer : quoi regarder précisément
# - ❓ Questions : réponses courtes, incisives
# - ⚠️ Attention : piège / erreur classique
# - 💡 Déclic : phrase d’ancrage


# **0 — Mise en place** 
#
# 🎯 **Objectif**
# Vérifier que l’environnement est stable : Kafka + topic + Spark.
#
# ✅ **À faire**
# - Kafka est lancé
# - Le topic `events_tp` existe (ou équivalent)
# - Vous savez produire quelques événements (sinon utilisez votre producer du TP2)
#
# ⚠️ **Attention**
# Un TP streaming échoue souvent par **mauvaise infrastructure**, pas par bug Python.

In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("TP3_Streaming_5h") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")


# **0.1 — Schéma et lecture Kafka**
#
# 🎯 **Objectif**
# Lire le flux **tel qu’il arrive**, avant de l’interpréter.
#
# 🔁 **Rappel**
# Un événement (dans ce TP) ≈
# - `user_id` (clé métier)
# - `amount` (montant)
# - `event_time` (timestamp métier)
#
# ⚠️ **Attention**
# Dans la vraie vie, le timestamp est parfois absent, incorrect, ou manipulable.
# Ici on suppose qu’il est fourni correctement (mais on discutera les risques).

In [ ]:

schema = StructType([
    StructField("user_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("event_time", TimestampType())
])

raw = spark.readStream.format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "events_tp") \
    .option("startingOffsets", "latest") \
    .load()

events = raw.selectExpr("CAST(value AS STRING) AS value_str") \
    .select(from_json(col("value_str"), schema).alias("data")) \
    .select("data.*")


#  **0.2 — Observation du flux brut**
#
# 🎯 **Objectif**
# Voir ce que vous traitez réellement.
#
# ✅ **À faire**
# Lancez l’affichage brut pendant 30–60 secondes.
#
# 🔬 **À observer**
# - l’ordre d’arrivée est-il stable ?
# - `event_time` est-il monotone ?
# - les clés `user_id` semblent-elles réparties ?
#
# ❓ **Questions**
# - Où est “le temps réel” : dans l’arrivée ou dans `event_time` ?
# - Un flux ressemble-t-il à un fichier ?
#
# 💡 **Déclic**
# Un flux n’a pas de fin : vous devez décider **quand une minute “est terminée”**.

In [ ]:

q_raw = events.writeStream.format("console").outputMode("append").start()


# ✅ **Action**
# Arrêtez `q_raw` après observation (sinon vous allez noyer votre console).
#
# ⚠️ **Attention**
# Beaucoup d’étudiants laissent 10 requêtes tourner en parallèle et concluent
# que “Spark est lent”. Non : c’est votre notebook qui est saturé.

In [ ]:
# Stop propre
try:
    q_raw.stop()
except Exception as e:
    print("Stop raw:", e)

# **1 — Le moteur peut mentir**
#
# 🎯 **Objectif**
# Comprendre que **Processing Time** peut fabriquer une réalité statistique fausse.
#
# 🧠 **Intuition**
# L’ordre d’arrivée n’est pas l’ordre métier.
#
# 🔁 **Rappel**
# | Horloge | Définition | Risque |
# |---|---|---|
# | Event Time | quand l’événement a eu lieu (métier) | stats fausses si ignorée |
# | Processing Time | quand le moteur traite | “temps réel” trompeur |
#
# ✅ **À faire**
# Lancer **deux pipelines** : l’un en Processing Time, l’autre en Event Time.
#
# 💡 **Déclic**
# Le pipeline peut être faux **sans bug**.

In [ ]:
# Pipeline Processing Time : window sur current_timestamp()
pt = events.groupBy(window(current_timestamp(), "1 minute").alias("w")) \
          .agg(count("*").alias("n"), sum("amount").alias("sum_amount"))

q_pt = pt.writeStream.format("console").outputMode("complete").start()

In [ ]:
# Pipeline Event Time : window sur event_time + watermark
et = events.withWatermark("event_time", "10 seconds") \
           .groupBy(window(col("event_time"), "1 minute").alias("w")) \
           .agg(count("*").alias("n"), sum("amount").alias("sum_amount"))

q_et = et.writeStream.format("console").outputMode("complete").start()

# 🔬 **À observer**
# - Les fenêtres et agrégats “PT” et “ET” sont-ils identiques ?
# - L’écart se réduit-il ou s’amplifie-t-il ?
#
# ❓ **Questions**
# - Lequel ment au métier ?
# - Dans quel contexte Processing Time serait acceptable ?
# - Si vous facturez un client “par minute”, quel temps devez-vous utiliser ?
#
# ⚠️ **Attention**
# Une erreur d’horloge n’est pas une erreur “numérique”.
# C’est une erreur **de décision** (vous optimisez le mauvais objectif).

In [ ]:

# Stop propre des requêtes de comparaison
for q in [q_pt, q_et]:
    try:
        q.stop()
    except Exception as e:
        print("Stop:", e)


# **1.1 — Rupture cognitive (expérience volontaire)**
#
# 🎯 **Objectif**
# Provoquer une divergence visible (même si votre flux est “propre”).
#
# ✅ **À faire**
# Modifiez votre producer (ou lancez un petit script)
# pour injecter **des événements avec event_time = maintenant - 90 secondes**.
#
# 🔬 **À observer**
# - le pipeline Event Time “rattrape” le passé
# - le pipeline Processing Time ne le fait pas
#
# ❓ **Questions**
# - Lequel reflète la réalité métier ?
# - Quel pipeline donnerait un dashboard “mensonger” au CEO ?
#
# 💡 **Déclic**
# Le moteur n’est pas “temps réel” par magie : il est “temps réel” par *contrat sur le temps*.

# **2 — Watermarks : décider du présent** 
#
# 🎯 **Objectif**
# Comprendre que le watermark est une **décision** : "je crois que le passé est terminé".
#
# 🧠 **Intuition**
# Le moteur doit fermer une fenêtre sans certitude.
#
# 🔁 **Rappel**
# Watermark typique = max_event_time - X  
# - X petit → rapide mais risque d’erreur  
# - X grand → précis mais plus lent
#
# ✅ **À faire**
# Comparer 3 watermarks : 2s, 10s, 60s.
#
# ⚠️ **Attention**
# Beaucoup de systèmes en production sont “optimisés latence” au prix de la vérité.

In [ ]:
def run_et_with_watermark(wm: str, label: str):
    df = events.withWatermark("event_time", wm) \
        .groupBy(window(col("event_time"), "1 minute").alias("w")) \
        .agg(count("*").alias("n"), sum("amount").alias("sum_amount")) \
        .select(lit(label).alias("wm_label"), col("w"), col("n"), col("sum_amount"))
    return df

et_wm_2s  = run_et_with_watermark("2 seconds",  "wm=2s")
et_wm_10s = run_et_with_watermark("10 seconds", "wm=10s")
et_wm_60s = run_et_with_watermark("60 seconds", "wm=60s")

union_wm = et_wm_2s.unionByName(et_wm_10s).unionByName(et_wm_60s)

q_wm = union_wm.writeStream.format("console").outputMode("complete").start()

# 🔬 **À observer**
# - Les résultats convergent-ils (à terme) ?
# - Lequel “publie” le plus vite ?
# - Lequel semble “stable” mais faux ?
#
# ❓ **Questions**
# - Si votre watermark est trop petit, que perdez-vous ?
# - Si votre watermark est trop grand, que perdez-vous ?
# - Quel watermark choisiriez-vous pour :
#   - (i) alertes sécurité immédiates
#   - (ii) reporting financier réglementaire
#
# 💡 **Déclic**
# Le watermark est un **contrat** entre vérité et délai.

In [ ]:
try:
    q_wm.stop()
except Exception as e:
    print("Stop wm:", e)

# **2.1 — Expérience “late events” (contrôlée)**
#
# 🎯 **Objectif**
# Voir les corrections (ou leur absence).
#
# ✅ **À faire**
# Injectez pendant 1 minute des événements dont `event_time` est :
# - (a) maintenant - 5s (normal)
# - (b) maintenant - 30s (tardif)
# - (c) maintenant - 3 minutes (très tardif)
#
# 🔬 **À observer**
# - Quels événements influencent encore les fenêtres ?
# - Quels événements sont “trop tard” (ignorés) ?
#
# ❓ **Questions**
# - Une correction tardive améliore-t-elle toujours le système ?
# - Un résultat corrigé 5 fois est-il crédible pour un décideur ?
#
# ⚠️ **Attention**
# Trop de corrections → instabilité des dashboards / alertes.

# **3 — Fenêtres : précision vs coût** 
#
# 🎯 **Objectif**
# Comprendre que le type de fenêtre change :
# - le nombre de calculs,
# - le volume de résultats,
# - la charge,
# - la lisibilité métier.
#
# 🧠 **Intuition**
# "Plus précis" veut souvent dire "plus cher".
#
# 🔁 **Rappel**
# - Tumbling : simple, non chevauchant
# - Sliding : chevauchement, plus de calculs
#
# ✅ **À faire**
# Comparer tumbling 1 min vs sliding 1 min / step 10s.

In [ ]:
tumbling = events.withWatermark("event_time", "10 seconds") \
    .groupBy(window(col("event_time"), "1 minute").alias("w")) \
    .agg(count("*").alias("n"), sum("amount").alias("sum_amount")) \
    .select(lit("tumbling(1m)").alias("win"), col("w"), col("n"), col("sum_amount"))

sliding = events.withWatermark("event_time", "10 seconds") \
    .groupBy(window(col("event_time"), "1 minute", "10 seconds").alias("w")) \
    .agg(count("*").alias("n"), sum("amount").alias("sum_amount")) \
    .select(lit("sliding(1m/10s)").alias("win"), col("w"), col("n"), col("sum_amount"))

cmp_windows = tumbling.unionByName(sliding)

q_win = cmp_windows.writeStream.format("console").outputMode("complete").start()

# 🔬 **À observer**
# - Lequel produit le plus de lignes de sortie ?
# - Lequel “rafraîchit” le plus souvent ?
# - Lequel est plus lisible ?
#
# ❓ **Questions**
# - Pourquoi sliding coûte plus cher (même si le flux est identique) ?
# - Dans quel cas sliding est indispensable ?
# - Dans quel cas sliding est un luxe coûteux ?
#
# ⚠️ **Attention**
# Un système peut “fonctionner” en sliding… jusqu’au jour où un pic de trafic le fait tomber.

In [ ]:
try:
    q_win.stop()
except Exception as e:
    print("Stop win:", e)

# **3.1 — Pause conceptuelle**
#
# ✅ À faire
# Écrivez 6 lignes maximum :
#
# - un exemple où tumbling est la bonne idée,
# - un exemple où sliding est la bonne idée,
# - et la raison “ingénieur système” (pas juste “c’est plus précis”).

# **4 — State : mémoire intelligente ou bombe (≈ 75 min)**
#
# 🎯 **Objectif**
# Comprendre que l’état est :
# - la source d’intelligence (compter, joindre, détecter)
# - et la source de catastrophe (mémoire infinie)
#
# 🧠 **Intuition**
# Sans état, le moteur ne comprend que le présent.
# Avec état, il comprend une histoire.
#
# 🔁 **Rappel**
# Keyed State ≈ un état par clé (ex : user_id → compteur)
#
# ⚠️ **Attention**
# Sans stratégie (TTL / fenêtre / bornage), l’état croît → crash “inévitable”.

In [ ]:
# Agrégation par clé : état potentiellement infini si la cardinalité des user_id est grande
per_user = events.groupBy(col("user_id")).agg(
    count("*").alias("n_events"),
    sum("amount").alias("sum_amount"),
    max("event_time").alias("last_event_time")
)

q_state = per_user.writeStream.format("console").outputMode("complete").start()

# 🔬 **À observer**
# - Le nombre de clés augmente-t-il ?
# - Les sorties deviennent-elles lourdes ?
#
# ❓ **Questions**
# - Pourquoi cette agrégation est-elle dangereuse ?
# - Que se passe-t-il si `user_id` a 10 millions de valeurs ?
# - Quelle serait une stratégie pour borner l’état ?
#
# 💡 **Déclic**
# Une opération “simple” (groupBy key) peut être la source d’une bombe mémoire.

In [ ]:
try:
    q_state.stop()
except Exception as e:
    print("Stop state:", e)

# **4.1 — Bornez l’état par le temps (fenêtre + watermark)**
#
# 🎯 **Objectif**
# Transformer un état potentiellement infini en état borné temporellement.
#
# ✅ **À faire**
# Refaire une agrégation par user_id **dans une fenêtre**.
#
# 💡 **Déclic**
# Fenêtre = bornage naturel de la mémoire.

In [ ]:
per_user_windowed = events.withWatermark("event_time", "20 seconds") \
    .groupBy(window(col("event_time"), "2 minutes").alias("w"), col("user_id")) \
    .agg(count("*").alias("n_events"), sum("amount").alias("sum_amount")) \
    .select(col("w"), col("user_id"), col("n_events"), col("sum_amount"))

q_state_bounded = per_user_windowed.writeStream.format("console").outputMode("append").start()

# 🔬 **À observer**
# - L’état “se vide-t-il” naturellement quand les fenêtres passent ?
# - La sortie est-elle plus stable ?
#
# ❓ **Questions**
# - Pourquoi cette version est plus “safe” en production ?
# - Quelle information métier perdez-vous (si vous fenêtnez) ?
#
# ⚠️ **Attention**
# Borner l’état résout la mémoire, mais peut casser des besoins métier (ex : historique long).

In [ ]:
try:
    q_state_bounded.stop()
except Exception as e:
    print("Stop bounded:", e)

# **4.2 — Mini-étude : “fraude” (stateful sur horizon glissant)**
#
# 🎯 **Objectif**
# Raisonner sur un cas métier qui nécessite de la mémoire.
#
# Contexte :
# - on veut détecter : “plus de 5 paiements en 2 minutes pour un user_id”
#
# ✅ **À faire**
# Construisez l’alerte à partir de la version fenêtrée.
#
# ❓ **Questions**
# - Est-ce du temps événement ou du temps traitement ?
# - Quel watermark minimum est acceptable ?
# - Que fait-on des late events (faut-il corriger une alerte) ?
#
# ⚠️ **Attention**
# En fraude, corriger tard peut être inutile : l’action doit être rapide.

In [ ]:
alerts = per_user_windowed.where(col("n_events") > 5) \
    .select(col("w").alias("window"), col("user_id"), col("n_events"), col("sum_amount"))

q_alerts = alerts.writeStream.format("console").outputMode("append").start()

In [ ]:
try:
    q_alerts.stop()
except Exception as e:
    print("Stop alerts:", e)

# **5 — Backpressure : le pipeline s’étouffe**
#
# 🎯 **Objectif**
# Provoquer une saturation **réaliste** et la diagnostiquer.
#
# 🧠 **Intuition**
# Un opérateur lent impose sa cadence à tout le pipeline.
#
# 🔁 **Rappel**
# Symptômes typiques :
# - lag Kafka qui augmente,
# - latence end-to-end qui explose,
# - CPU anormal sur un opérateur,
# - files internes qui gonflent.
#
# ✅ **À faire**
# On va simuler un sink lent.
#
# ⚠️ **Attention**
# La saturation ne se “corrige” pas avec une ligne magique.
# On **rééquilibre** un système.

In [ ]:
# On simule un sink lent via foreachBatch (plus propre que foreach row).
def slow_sink(batch_df, batch_id):
    # Simule une écriture aval lente
    time.sleep(2.0)  # <- ralentisseur
    # "écriture" fictive : on force une action
    batch_df.count()

slow_pipeline = events.withWatermark("event_time", "10 seconds") \
    .groupBy(window(col("event_time"), "30 seconds").alias("w")) \
    .agg(count("*").alias("n"), sum("amount").alias("sum_amount"))

q_slow = slow_pipeline.writeStream \
    .foreachBatch(slow_sink) \
    .outputMode("complete") \
    .start()

# 🔬 **À observer**
#
# - Le job devient-il “en retard” ?
# - Votre producer continue-t-il à produire ?
# - Le système “absorbe-t-il” ou “s’écroule-t-il” ?
#
# ❓ **Questions (diagnostic)**
# - Quel est le goulot ici ?
# - Qu’est-ce qui augmente : latence, backlog, ou les deux ?
# - Quelle métrique est la plus “vitale” à surveiller ?
#
# ⚠️ **Attention**
# Dans un vrai système, la backpressure remonte en amont :
# ce qui était “stable” devient instable par cascade.

In [ ]:
try:
    q_slow.stop()
except Exception as e:
    print("Stop slow:", e)

# **5.1 — Stabilisation**
#
# 🎯 **Objectif**
# Tester des correctifs et **classer** leur efficacité.
#
# ✅ **À faire**
# Proposez au moins 3 actions parmi :
# 1) augmenter le parallélisme (si applicable),
# 2) simplifier les fenêtres (sliding → tumbling),
# 3) réduire la fréquence de publication,
# 4) changer le partitionnement (côté Kafka / clé),
# 5) optimiser l’écriture sink (batching, async, buffer),
# 6) réduire le volume (filtrer / sampler),
# 7) séparer le pipeline (découplage).
#
# ❓ **Questions**
# - Quelle action traite la **cause** ?
# - Quelle action masque seulement le **symptôme** ?
# - Quelle action augmente le coût infra ?
#
# 💡 **Déclic**
# On ne “corrige” pas un flux : on **rééquilibre** un système.

# **6 — Diagnostic global**
#
# 🎯 **Objectif**
# Savoir lire un pipeline comme un système.
#
# ✅ **À faire**
# Remplissez les 6 lignes suivantes (phrase courte et précise !) :
#
# 1) Le plus grand risque de Processing Time est : …
# 2) Un watermark est : …
# 3) Sliding window coûte plus cher car : …
# 4) L’état est dangereux parce que : …
# 5) La backpressure signifie : …
# 6) Mon indicateur n°1 à monitorer est : …

# **7 — Challenge final** 
#
# 🎯 **Objectif**
# Concevoir un pipeline streaming réaliste et justifié.
#
# ## Cas : Paiements mondiaux en temps réel
#
# Contexte :
# - utilisateurs sur 5 continents
# - réseaux mobiles instables
# - pics de trafic ×30 (Black Friday)
# - latence max acceptable pour dashboard : 2 secondes
# - détection fraude doit réagir en < 30 secondes
# - reporting réglementaire doit être exact (même si plus lent)
#
# ✅ **À faire**
# Concevez deux vues :
#
# **A) Vue “temps réel opérationnel”** (latence très faible)
# - destinée à l’action immédiate
#
# **B) Vue “vérité métier / réglementaire”** (plus lente mais plus exacte)
# - destinée au reporting
#
# ❓ **À justifier (obligatoire)**
# - Event Time ou Processing Time (pour A et pour B) ?
# - stratégie watermark (A vs B) ?
# - type de fenêtre (A vs B) ?
# - gestion des late events (corriger / ignorer / route spéciale) ?
# - état : comment borner (fenêtre, TTL, design) ?
# - stratégie anti-backpressure (priorités, découplage, scaling) ?
#
# ⚠️ **Attention**
# “Exactly-once” n’est pas magique : il dépend du pipeline complet (source + moteur + sink).
#
# 💡 **Déclic final**
# Un moteur de streaming est un **système temporel vivant distribué**.

# **8 — Cellule utilitaire (arrêt général des streams)**
#
# ✅ **À faire**
# Si vous avez lancé plusieurs requêtes, utilisez cette cellule pour tout arrêter proprement.
#
# ⚠️ **Attention**
# Ne laissez pas 10 requêtes actives en même temps : vous masquez les phénomènes.

In [ ]:
# Stop all active streams
for q in spark.streams.active:
    try:
        q.stop()
    except Exception as e:
        print("Stop stream:", e)

print("Active streams:", len(spark.streams.active))

# 🧠 **Dernière phrase (à retenir)**
#
# > Un pipeline temps réel n’est pas “du code qui tourne”.  
# > C’est une décision continue sur : **quand**, **quoi**, **combien**, et **à quel prix**.